In [ ]:
import json
from PIL import Image
from torch.utils.data import Dataset

class ImageNet100Dataset(Dataset):
    def __init__(self, root_dir, indices=None):
        self.root_dir = root_dir
        with open(f"{root_dir}/labels.json") as f:
            self.labels = json.load(f)
        
        self.indices = indices if indices is not None else list(range(len(self.labels)))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        img = Image.open(f"{self.root_dir}/{real_idx}.jpg").convert("RGB")
        label = self.labels[real_idx]
        return img, label

In [ ]:
import json
import random

from matplotlib.pylab import tri

with open("../data/labels.json") as f: 
    labels = json.load(f) 
with open("../data/class_names.json") as f: # optional, just for reference
    class_names = json.load(f)  

SEED = 42
all_classes = list(range(100))
random.Random(SEED).shuffle(all_classes)

holdout_classes = set(all_classes[:20])
train_classes = set(all_classes[20:])

train_indices = [i for i, lbl in enumerate(labels) if lbl in train_classes]
holdout_indices = [i for i, lbl in enumerate(labels) if lbl in holdout_classes]

print(f"Train images: {len(train_indices)}")    
print(f"Holdout images: {len(holdout_indices)}")

train_base = ImageNet100Dataset("../data/imagenet100_128px", indices=train_indices)
holdout_base = ImageNet100Dataset("../data/imagenet100_128px", indices=holdout_indices)

train_base[0]

In [ ]:
from torch.utils.data import Dataset, DataLoader


class JigsawDataset(Dataset): # Datset for jigsaw puzzle task
    """
    """

    def __init__(self, image_dataset, permutation_set, seed=None):
        self.image_dataset = image_dataset 
        self.permutation_set = permutation_set
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.image_dataset)

    def __getitem__(self, idx):
        img, label = self.image_dataset[idx]
        perm_index = self.rng.randint(0, len(self.permutation_set) - 1)
        perm = self.permutation_set[perm_index]
        shuffled_patches = [img.crop((x, y, x + 32, y + 32)) for (x, y) in perm]
        
        return shuffled_patches, perm_index

In [1]:
import os
print(f"Cpu count: {os.cpu_count()}")

Cpu count: 12
